In [ ]:
#instalação de bibliotecas
import cv2
import os

class pipeline_Processor:
    """
    Pipeline de pré-processamento para peças de fundição (Casting Dataset) para peças de fundição (Casting Dataset).
    Processa UMA imagem por vez, seguindo as sprints do mini-projeto:
    Grayscale-> Blur -> Threshold (Otsu)  -> Morfologia  -> Bordas (Canny) -> Resize
    """
    def __init__(self,blur_kernel=(5,5),canny_thresh1=50,canny_thresh2=150,
                 morph_kernel_size=(3,3),output_size=(256,256)):
        self.blur_kernel=blur_kernel
        self.canny_thresh1=canny_thresh1
        self.canny_thresh2=canny_thresh2
        self.morph_kernel=cv2.getStructuringElement(cv2.MORPH_ELLIPSE,morph_kernel_size)
        self.output_size=output_size


    # 1.Mostrar a imagem
    def _show_image(self,img,file_name="Image"):
        try:
            cv2.imshow(file_name,img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
        except Exception as e:
            print(f"[Erro ao exibir imagem]: {e}")

    # 2. Leitura de imagem do repósitório    
    def _read_image(self,path):
        try:
            img=cv2.imread(path)
            if img is None:
                print(f"[Aviso] Imagem não carregada ou corrompida : {path}")
            return img
        except Exception as e:
            print(f"[Erro ao exibir imagem]: {e}")
    """
    Sprint 3: conversão para escala de cinza (Grayscale) e aplicação de filtro para redução de ruído (ex: Gaussian Blur ou Median Blur).
    """
    #3. Conversão Para Escala de cinza 
    def _to_gray_scale(self,img):
        try:
            return cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
        except Exception as e:
            print(f"[Erro na conversão para escala de cinza] : {e}")
            return None
        
    #4. Filtro para redução de ruido 
    def _apply_blur(self,img):
        try:
            return cv2.GaussianBlur(img,self.blur_kernel,0)
        except Exception as e:
            print(f"[Erro na  suavização gaussiana] : {e}")
            return None

    """
    - **Sprint 4 -  Aplica técnicas de limiarização (Thresholding, como o método de Otsu) e detecção de bordas (ex: Canny ou Sobel) para destacar os contornos da peça e possíveis falhas.
    """
    #5.Limiarização com Otsu
    def _apply_threshold_otsu(self,img):
        try:
            _,binary=cv2.threshold(img,0,255,cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            return binary
        except Exception as e:
            print(f"[Erro na  limiarização] : {e}")
            return None

    #6. Detecção de bordas com Canny
    def _apply_canny(self,img):
        try:
            return cv2.Canny(img,self.canny_thresh1,self.canny_thresh2)
        except Exception as e:
            print(f"[Erro na  detecção de bordas] : {e}")
            return None

    """
     **Sprint 5: Utiliza operações morfológicas (Erosão/Dilatação) para remover ruídos da segmentação e redimensionar (Resize) todas as imagens para um tamanho padrão (ex: 256x256 pixels).
    """
    #7. Operações Morfológicas
    def _apply_morphology(self,img):
        try:
            #Opened: erosão, seguido de dilatacao, remoção de ruidos pequenos
            opened=cv2.morphologyEx(img,cv2.MORPH_OPEN, self.morph_kernel)
            #Closed: dilatacao seguida de erosão, fecha buracos pequenos da peça 
            closed=cv2.morphologyEx(opened,cv2.MORPH_CLOSE,self.morph_kernel)
            return closed
        except Exception as e:
            print(f"[Erro na etapa de operações morfológicas]: {e}")
            return None

    #8. Redimensionamento padrão
    def _resize_image(self,img):
        try:
            return cv2.resize(img,self.output_size,interpolation=cv2.INTER_AREA)
        except Exception as e:
            print(f"[Erro no redimensionamento padrão]: {e}")
            return None

    """
    *Sprint 6 - Salva as imagens processadas no diretório de saída, consolida a branch no GitHub,
    """
    #9: Salva imagem
    def _save_image(self,img,path):
        try:
            cv2.imwrite(path,img)
            return True
        except Exception as e:
            print(f"[Erro ao salvar imagem]: {e}")
            return None

    #10 Orquestração de pipeline para uma imagem
    def process_single_image(self, input_path, output_path, mostrar=False):
        # 1. Leitura
        img = self._read_image(input_path)
        if img is None:
            return False

        # 2. Grayscale
        gray = self._to_gray_scale(img)
        if gray is None:
            return False

        # 3. Blur
        blurred = self._apply_blur(gray)
        if blurred is None:
            return False

        # 4. Threshold com Otsu
        binary = self._apply_threshold_otsu(blurred)
        if binary is None:
            return False

        # 5. Detecção de bordas (Canny sobre a imagem suavizada)
        edges = self._apply_canny(blurred)
        if edges is None:
            return False

        # 6. Morfologia sobre as bordas (limpa ruídos e fecha falhas)
        edges_clean = self._apply_morphology(edges)
        if edges_clean is None:
            return False

        # 7. Resize
        final = self._resize_image(edges_clean)
        if final is None:
            return False

        # Mostrar etapas (opcional)
        if mostrar:
            self._show_image(img, "1 - Original")
            self._show_image(gray, "2 - Grayscale")
            self._show_image(blurred, "3 - Gaussian Blur")
            self._show_image(binary, "4 - Threshold Otsu")
            self._show_image(edges, "5 - Bordas Canny")
            self._show_image(edges_clean, "6 - Morfologia")
            self._show_image(final, "7 - Resultado Final")

        # Salvar resultado
        return self._save_image(final, output_path)

In [ ]:
#Instanciando a classe para chamar e mostrar/salvar a imagem

pipeline=pipeline_Processor()
# pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_k7.jpeg')
# pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_k5.jpeg')
#pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeD ASd g','output/processed_images/test/cast_ok_0_3136_canny_120_240.jpeg')
#pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_canny_30_100.jpeg')
#pipeline.process_single_image('data/raw_images/ok_front/cast_ok_0_3136.jpeg','output/processed_images/test/cast_ok_0_3136_canny_50_150.jpeg')
#pipeline.process_single_image('data/raw_images/def_front/cast_def_0_2234.jpeg','output/processed_images/test/cast_def_0_2234_canny_120_240.jpeg')
pipeline.process_single_image('data/raw_images/def_front/cast_def_0_2234.jpeg','output/processed_images/test/cast_def_0_2234_test.jpeg')


#testando pipeline imagem unica
# processador = pipeline_Processor()
# caminho_entrada = "data/raw_images/def_front/cast_def_0_9477.jpeg"
# caminho_saida = "data/processed_images/test/cast_def_0_9477_processada.jpeg"
# processador.process_single_image(
#      caminho_entrada,
#      caminho_saida,
#     mostrar=True
# )
#image=pipeline._read_image('data/raw_images/def_front/cast_def_0_146.jpeg')


#testando images
# img_cinza=pipeline._to_gray_scale(image)
# img_blur=pipeline._apply_blur(img_cinza)
# img_thresh=pipeline._apply_threshold_otsu(img_blur)
# img_bordas=pipeline._apply_canny(img_thresh)
# img_morfologia=pipeline._apply_morphology(img_bordas)
# img_resize=pipeline._resize_image(img_morfologia)

# pipeline._save_image(img_resize,'output/processed_iamges/test/cast_def_0_146.jpeg')
#pipeline._show_image(img_resize,'cast_def_0_146.jpeg_resize')
#pipeline._show_image(image,'cast_def_0_146.jpeg_original')0
# pipeline._read_image('data/raw_images/def_front/cast_def_0_146.jpeg')


#pipeline unico

True